<a href="https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/inference_example1.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# MarinFold Inference Heatmap Demo

This notebook clones or reuses the repo, installs the packaged `marinfold` library with the right optional extra for the current hardware, runs the current `1B` or `1.5B` model on an mmCIF from RCSB, and plots ground-truth vs predicted distance heatmaps inline.


## Before You Run

Runtime selection is automatic:

- NVIDIA GPU -> MarinFold `vllm`
- Apple Silicon -> MarinFold `mlx`
- CPU / other -> MarinFold `transformers`
- TPU -> detected, but this notebook stops with a clear message because MarinFold does not currently expose a TPU inference backend

If you are in Colab and want the fastest path, switch to a **GPU** runtime. After the install cell, Colab may ask for a runtime restart before the imported backend packages are visible; if imports fail right after install, restart the session and rerun the notebook.

On smaller Colab GPUs such as a T4, prefer the `1B` model first. If you switch to `1.5B` and hit memory pressure, lower `BATCH_SIZE`.


In [ ]:
# @title Configuration
MODEL_NICKNAME = "1B"  # @param ["1B", "1.5B"]
PDB_ID = "1QYS"  # @param {type:"string"}
QUERY_ATOM = "CA"  # @param ["CA", "CB"]
BACKEND_MODE = "auto"  # @param ["auto", "vllm", "mlx", "transformers"]
DTYPE = "float16"  # @param ["float16", "bfloat16"]
BATCH_SIZE = 32  # @param {type:"integer"}
GPU_MEMORY_UTILIZATION = 0.8  # @param {type:"number"}

print(
    {
        "model": MODEL_NICKNAME,
        "pdb_id": PDB_ID.upper(),
        "query_atom": QUERY_ATOM,
        "backend_mode": BACKEND_MODE,
        "dtype": DTYPE,
        "batch_size": BATCH_SIZE,
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
    }
)


The shipped `marinfold evaluate` path supports direct `CA` or `CB` queries. `CA` is the safest default for a quick notebook run; if you choose `CB`, any GLY or missing-CB pairs will be masked out of the plot.


In [ ]:
import os
import platform
import shutil
import subprocess
from pathlib import Path


def _find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "marinfold" / "pyproject.toml").exists():
            return candidate
    return None


def _has_nvidia_gpu() -> bool:
    if shutil.which("nvidia-smi") is None:
        return False
    result = subprocess.run(
        ["nvidia-smi", "-L"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    return result.returncode == 0 and bool(result.stdout.strip())


def _has_tpu() -> bool:
    return any(
        os.environ.get(key)
        for key in ("COLAB_TPU_ADDR", "TPU_NAME", "TPU_WORKER_ID")
    )


def _is_apple_silicon() -> bool:
    return platform.system() == "Darwin" and platform.machine() == "arm64"


def _auto_backend_choice():
    if _has_nvidia_gpu():
        return {
            "backend": "vllm",
            "extra": "vllm",
            "runtime": "nvidia-gpu",
            "note": "Using vLLM because an NVIDIA GPU is available.",
        }
    if _has_tpu():
        return {
            "backend": None,
            "extra": None,
            "runtime": "tpu",
            "note": "TPU detected, but MarinFold does not currently expose a TPU inference backend in this notebook.",
        }
    if _is_apple_silicon():
        return {
            "backend": "mlx",
            "extra": "mlx",
            "runtime": "apple-silicon",
            "note": "Using MLX because Apple Silicon is available.",
        }
    return {
        "backend": "transformers",
        "extra": "transformers",
        "runtime": "cpu-or-other",
        "note": "Falling back to transformers because no supported accelerator was detected.",
    }


BASE_WORKDIR = Path("/content") if Path("/content").exists() and os.access("/content", os.W_OK) else Path.cwd()
REPO_DIR = _find_repo_root(Path.cwd()) or (BASE_WORKDIR / "MarinFold")
if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Open-Athena/MarinFold.git",
            str(REPO_DIR),
        ],
        check=True,
    )

AUTO_BACKEND = _auto_backend_choice()
if BACKEND_MODE == "auto":
    BACKEND_NAME = AUTO_BACKEND["backend"]
    EXTRA_NAME = AUTO_BACKEND["extra"]
else:
    BACKEND_NAME = BACKEND_MODE
    EXTRA_NAME = BACKEND_MODE

print(f"repo checkout: {REPO_DIR}")
print({
    "backend": BACKEND_NAME,
    "extra": EXTRA_NAME,
    "runtime": AUTO_BACKEND["runtime"],
    "note": AUTO_BACKEND["note"],
})

if BACKEND_NAME is None:
    raise RuntimeError(AUTO_BACKEND["note"])


In [ ]:
import importlib.util
import shutil
import subprocess
import sys

PACKAGE_DIR = REPO_DIR / "marinfold"
if importlib.util.find_spec("pip") is not None:
    install_cmd = [sys.executable, "-m", "pip", "install", "-q", "-e", f".[{EXTRA_NAME}]"]
elif shutil.which("uv") is not None:
    install_cmd = ["uv", "pip", "install", "--python", sys.executable, "-q", "-e", f".[{EXTRA_NAME}]"]
else:
    raise RuntimeError("Neither pip nor uv is available to install notebook dependencies.")

subprocess.run(
    install_cmd,
    cwd=PACKAGE_DIR,
    check=True,
)
print(f"installed marinfold extra: {EXTRA_NAME}")


In [ ]:
import importlib
import json
import os
import platform
import time
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np

from marinfold import write_eval
from marinfold.document_structures.contacts_and_distances_v1 import (
    InferenceConfig,
    evaluate,
    plot_evaluate_pdf,
)
from marinfold.registry import list_model_entries, resolve_model_entry

os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")

runtime_summary = {
    "backend": BACKEND_NAME,
    "runtime": AUTO_BACKEND["runtime"],
    "platform": platform.platform(),
    "machine": platform.machine(),
}

torch_spec = importlib.util.find_spec("torch")
if torch_spec is not None:
    import torch

    runtime_summary["torch_version"] = torch.__version__
    runtime_summary["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        runtime_summary["gpu_name"] = props.name
        runtime_summary["gpu_memory_gib"] = round(props.total_memory / (1024 ** 3), 1)

print(runtime_summary)


In [ ]:
entries = {entry.nickname: entry for entry in list_model_entries()}
print("Available model nicknames:", sorted(entries))

entry = resolve_model_entry(MODEL_NICKNAME)
print(f"Using model: {entry.nickname}")
print(f"HF URL: {entry.url}")
if entry.wandb_url:
    print(f"W&B: {entry.wandb_url}")


In [ ]:
RUN_DIR = BASE_WORKDIR / "marinfold_notebook_runs"
RUN_DIR.mkdir(parents=True, exist_ok=True)
INPUT_PATH = RUN_DIR / f"{PDB_ID.upper()}.cif"
cif_url = f"https://files.rcsb.org/download/{PDB_ID.upper()}.cif"

if not INPUT_PATH.exists():
    urlretrieve(cif_url, INPUT_PATH)

print(f"downloaded: {INPUT_PATH}")
print(f"source url: {cif_url}")


In [ ]:
cfg = InferenceConfig(
    model=MODEL_NICKNAME,
    input_path=INPUT_PATH,
    backend=BACKEND_NAME,
    query_atom=QUERY_ATOM,
    batch_size=BATCH_SIZE,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
)

t0 = time.time()
result = evaluate(cfg)
elapsed = time.time() - t0

print(json.dumps(result.metrics, indent=2))
print(f"elapsed_seconds: {elapsed:.1f}")


In [ ]:
DISTANCE_MAX_A = 32.0


def reconstruct_matrix(rows, value_key, n_residues):
    out = np.full((n_residues, n_residues), np.nan, dtype=np.float32)
    np.fill_diagonal(out, 0.0)
    for row in rows:
        i = int(row["i"]) - 1
        j = int(row["j"]) - 1
        value = float(row[value_key])
        out[i, j] = value
        out[j, i] = value
    return out


rows = result.per_example
if not rows:
    raise ValueError("No per-example rows were produced.")

entry_id = rows[0]["entry_id"]
n_residues = int(result.extras["per_structure_n_residues"][entry_id])
n_pairs = int(result.extras["per_structure_n_pairs"][0][entry_id])
mae = float(result.metrics["mae_at_n0_angstrom"])

gt = reconstruct_matrix(rows, "gt_angstrom", n_residues)
pred = reconstruct_matrix(rows, "expected_angstrom", n_residues)
abs_err = np.abs(pred - gt)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))

im0 = axes[0].imshow(gt, vmin=0, vmax=DISTANCE_MAX_A, cmap="viridis")
axes[0].set_title(f"Ground truth {QUERY_ATOM}-{QUERY_ATOM} (Å)")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(pred, vmin=0, vmax=DISTANCE_MAX_A, cmap="viridis")
axes[1].set_title(f"Predicted {QUERY_ATOM}-{QUERY_ATOM} (Å)")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(abs_err, vmin=0, vmax=10.0, cmap="magma")
axes[2].set_title(f"|residual| (MAE = {mae:.2f} Å)")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xlabel("residue j")
    ax.set_ylabel("residue i")

fig.suptitle(
    f"{entry_id} | model={MODEL_NICKNAME} | atom={QUERY_ATOM} | pairs={n_pairs:,}",
    fontsize=13,
)
fig.tight_layout()
plt.show()


In [ ]:
OUTPUT_DIR = RUN_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

safe_model = MODEL_NICKNAME.replace(".", "_")
base_name = f"{entry_id}_{safe_model}_{QUERY_ATOM.lower()}"

metrics_path = OUTPUT_DIR / f"{base_name}_metrics.json"
plots_pdf_path = OUTPUT_DIR / f"{base_name}_plots.pdf"
heatmap_png_path = OUTPUT_DIR / f"{base_name}_heatmap.png"

write_eval(metrics_path, result, structure_name="contacts-and-distances-v1")
plot_evaluate_pdf(plots_pdf_path, result)
fig.savefig(heatmap_png_path, dpi=150, bbox_inches="tight")

print(f"saved metrics: {metrics_path}")
print(f"saved pdf: {plots_pdf_path}")
print(f"saved png: {heatmap_png_path}")
